In [ ]:
import pandas as pd
import wandb
from tqdm import tqdm

from utils_modelo_reviews.preprocesamiento import read_reviews, clean_text
from fastopic import FASTopic
from topmost import Preprocess
from nltk.corpus import stopwords
import spacy
import re
from nltk.tokenize import RegexpTokenizer
import nltk
import numpy as np
import torch
nltk.download('punkt_tab')      
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng') 


c:\Users\imzli\Documents\Proyecto de Datos\c2526-R4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\imzli\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [2]:
df = read_reviews()

In [3]:
df.head(100)

,appid,is_positive,weight,text,language
1,747660,True,0.500000,nothing,en
3,747660,True,0.500000,"so, i completed the game and got all the achie...",en
4,747660,True,0.500000,its very good and worth the 40 bucks in my opi...,en
8,747660,True,0.500000,i like that its fun and has a lot of gameplay ...,en
10,747660,True,0.500000,holds a special place in my heart,en
...,...,...,...,...,...
143,747660,True,0.500000,fun and scary,en
146,747660,False,0.527076,"jst not good dont buy/gen, just watch someone ...",en
147,747660,True,0.500000,great game! i have 30 hours of game play on it...,en
148,747660,True,0.500000,the best game ever super fun and very worth it!!!,en


In [4]:
tqdm.pandas(desc="Limpiando texto")

In [5]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

In [ ]:
stop_words = set(stopwords.words('english'))

tokenizer = RegexpTokenizer(r'\w+')
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def fast_noun_extractor(texts):
    cleaned = []

    for doc in tqdm(nlp.pipe(texts, n_process=1, batch_size=1000), total=len(texts)):
        tokens = [t.lemma_.lower() for t in doc if t.is_alpha and t.pos_ == "NOUN"]
        cleaned.append(" ".join(tokens))
    return cleaned

df["text_clean"] = fast_noun_extractor(df["text"].copy().astype(str).tolist())
df = df[df["text_clean"].str.split().str.len() >= 5].copy()

100%|██████████| 205883/205883 [22:13<00:00, 154.44it/s]


In [7]:
preprocess = Preprocess(vocab_size=3000, min_doc_count=750, max_doc_freq=0.3, stopwords=stop_words)
model = FASTopic(num_topics = 8, verbose= True, preprocess=preprocess,device="cuda", low_memory=True, low_memory_batch_size=50000,DT_alpha = 5,TW_alpha = 5)

2026-04-06 00:56:08,430 - FASTopic - use device: cuda


In [8]:
def remove_hearts(text):
    text = re.sub(r'(hearts){2,}', '', text)
    text = re.sub('heartsheart', '', text)
    text = re.sub('heart', '', text)
    return text
df["text_clean"] = df["text_clean"].progress_apply(remove_hearts)

Limpiando texto: 100%|██████████| 115168/115168 [00:00<00:00, 160365.77it/s]


In [ ]:
topic_top_words, doc_topic_dist = model.fit_transform(df["text_clean"].to_list(),learning_rate=0.02)

2026-04-06 01:10:52,388 - FASTopic - Using low memory mode.
2026-04-06 01:10:52,388 - FASTopic - Fine-tuning the model.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1131.99it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
parsing texts: 100%|██████████| 115168/115168 [00:03<00:00, 33889.90it/s]
c:\Users\imzli\Documents\Proyecto de Datos\c2526-R4\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'

2026-04-06 01:11:02,927 - TopMost - Real vocab size: 594
2026-04-06 01:11:02,929 - TopMost - Real training size: 115168 	 avg length: 16.867


In [ ]:
df["Topic_ID"] = np.argmax(doc_topic_dist, axis=1)

topic_tags = {
    0: "Price & Value",
    1: "Design & Immersion",
    2: "Story & Characters",
    3: "Arcade & Shooters",
    4: "Combat & Difficulty",
    5: "Controls & UI",
    6: "Community & Updates",
    7: "Art & Soundtrack"
}

df["Topic_Name"] = df["Topic_ID"].map(topic_tags)

print(df["Topic_Name"].value_counts())

Topic_Name
Controls & UI          19745
Price & Value          17793
Art & Soundtrack       16606
Combat & Difficulty    16201
Arcade & Shooters      12118
Story & Characters     11590
Community & Updates    10908
Design & Immersion     10207
Name: count, dtype: int64


In [11]:
pd.set_option('display.max_colwidth', 500)
df

,appid,is_positive,weight,text,language,text_clean,Topic_ID,Topic_Name
3,747660,True,0.500000,"so, i completed the game and got all the achievements 100%, even though i'm a fan of fnaf, i didn't think i'd complete it in 3 days.",en,game achievement fan fnaf day,2,Story & Characters
4,747660,True,0.500000,"its very good and worth the 40 bucks in my opinion. warehouse is hard, but beatable, and the maze in mazercise is like impossible. had to watch vid. oerall i love this",en,buck opinion warehouse maze mazercise oerall,6,Community & Updates
8,747660,True,0.500000,i like that its fun and has a lot of gameplay and content i dont like that if you dont have enough video card memory it crashes when you open certain cams and is laggy,en,fun lot gameplay content video card memory cam,4,Combat & Difficulty
22,747660,True,0.500000,"its very lovable game with very wasted potential. if steelwool would not rush it and work on it for year or two it would be banger. but since they did we got 20% of the content this game supposed to have and very unoptimized buggy mess. also if u look for a scary horror, this game is not for you. maybe ruin is scary a little bit. animatronics are the best this franchise ever had, all of them are very lovable with so much unique personality. roxanne wolf. amazing graphics, game looks absolute...",en,game potential steelwool year banger content game mess horror game ruin bit animatronic franchise personality graphic game atmosphere game ruin animatronic fan fnaf world model animatronic game setting time bit map feature map quality price hint detail tutorial level game time loop detail place content mod game antagonist screen time game review game love flaw exception time sequel game cassie story game time,5,Controls & UI
24,747660,False,0.538462,absolutely atrocious port. how many years has it been? and this game is still riddled with bugs and glitches. i'd much rather play switch version than this. such a shame that this is the quality we get now.,en,port year game bug glitch switch version shame quality,0,Price & Value
...,...,...,...,...,...,...,...,...
261260,1848440,True,0.525862,"very good game, pretty difficult at some points. likeable characters, nice art style and superb voice acting. worth every penny!",en,game point character art style superb voice penny,5,Controls & UI
261263,2198070,False,0.500000,"i really like this concept and this is theoretically really fun, but in practice the abundance of hard-to-play, not-very-beneficial cards (i.e. water treatment plants and memorial parks) and lack of useful ones (i.e. roads) makes it challenging in a distinctly un-fun way.",en,concept practice abundance play card water treatment plant park lack one road way,4,Combat & Difficulty
261265,408410,True,0.500000,i've played through this game roughly twice by now (excluding dlc as of writing). it's quite entertaining and can be pretty difficult to manage quick actions at times when you have to make quick choices about what's the highest threat on the field at a moment's notice. the freeform tower-defense style of this game makes for a great time. the story's not much of a story and just dressing for the carnage. what really sells the game is the fully active involvement akin to games like sanctum 2 (...,en,game dlc writing action time choice threat field moment notice tower defense style game time story story carnage game involvement game sanctum degree minigun monkey btd tool ship way laser bomber tower upgrade game sense wall tower damage,7,Art & Soundtrack
261269,3300800,True,0.500000,"great game, small developer, cheap game, but really enjoyable and probably one of the better ""modern"" puzzle games. highly recommend.",en,game developer game puzzle game,0,Price & Value


In [12]:
doc_topic_dist

array([[0.1233506 , 0.14486237, 0.16311099, ..., 0.10774896, 0.10098467,
        0.10533934],
       [0.12448712, 0.12894815, 0.13399523, ..., 0.12382945, 0.14006089,
        0.12057911],
       [0.14068827, 0.12540434, 0.14049985, ..., 0.10315534, 0.10875016,
        0.09556495],
       ...,
       [0.11248395, 0.12829183, 0.12057956, ..., 0.12938249, 0.11764219,
        0.14914708],
       [0.14148718, 0.13919367, 0.11974777, ..., 0.12352291, 0.12371628,
        0.11967795],
       [0.10122917, 0.13459347, 0.12811725, ..., 0.16487464, 0.14143352,
        0.11287888]], shape=(115168, 8), dtype=float32)

In [13]:
model.get_topic(topic_idx=4)

(('player', np.float32(0.15765692)),
 ('friend', np.float32(0.099672474)),
 ('people', np.float32(0.08646594)),
 ('card', np.float32(0.084142454)),
 ('team', np.float32(0.0603593)))

In [14]:
fig = model.visualize_topic(top_n=10)
fig.show()

In [15]:
fig = model.visualize_topic_hierarchy()
fig.show()

c:\Users\imzli\Documents\Proyecto de Datos\c2526-R4\.venv\Lib\site-packages\fastopic\_plot.py:265: ClusterWarning:

The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix



In [16]:
fig = model.visualize_topic_weights(top_n=20, height=500)
fig.show()